In [ ]:
import json
import polars as pl

In [ ]:
with open(
    "/home/daniilsan/projects/MSc-Thesis/data/flow_img/train/annotations.json"
) as f:
    flow_val_data = json.load(f)
    train_df = pl.DataFrame(flow_val_data["annotations"])

    train_df.show(10)

with open(
    "/home/daniilsan/projects/MSc-Thesis/data/flow_img/val/annotations.json"
) as f:
    flow_val_data = json.load(f)
    val_df = pl.DataFrame(flow_val_data["annotations"])

    val_df.show(10)

In [ ]:
train_df = train_df.with_columns(
    pl.when(pl.col("area") < 1024)
    .then(pl.lit("small"))
    .when(pl.col("area") < 9216)
    .then(pl.lit("medium"))
    .otherwise(pl.lit("large"))
    .alias("size")
)

val_df = val_df.with_columns(
    pl.when(pl.col("area") < 1024)
    .then(pl.lit("small"))
    .when(pl.col("area") < 9216)
    .then(pl.lit("medium"))
    .otherwise(pl.lit("large"))
    .alias("size")
)

In [ ]:
# Plot a pie chart of the size distribution in val and train sets
import matplotlib.pyplot as plt

size_df_train = train_df.group_by("size").len().sort("size")
size_df_val = val_df.group_by("size").len().sort("size")
size_df_combined = (
    pl.concat([size_df_train, size_df_val]).group_by("size").sum().sort("size")
)

plt.figure(figsize=(16, 6))

plt.subplot(1, 3, 1)
plt.pie(size_df_train["len"], labels=size_df_train["size"], autopct="%1.1f%%")  # type: ignore
plt.title("Distribution of Object Sizes in Train Set")

plt.subplot(1, 3, 2)
plt.pie(size_df_val["len"], labels=size_df_val["size"], autopct="%1.1f%%")  # type: ignore
plt.title("Distribution of Object Sizes in Val Set")

plt.subplot(1, 3, 3)
plt.pie(size_df_combined["len"], labels=size_df_combined["size"], autopct="%1.1f%%")  # type: ignore
plt.title("Distribution of Object Sizes in Combined Set")

plt.show()

## Analysis of the detection results of the AMVI (Aggregated Multi-View Inference)


In [ ]:
import matplotlib.pyplot as plt

df = pl.read_parquet("/tmp/cli_test_candidates.parquet")

small_df = df.filter(pl.col("size_bin") == "small")
medium_df = df.filter(pl.col("size_bin") == "medium")
large_df = df.filter(pl.col("size_bin") == "large")

plt.figure(figsize=(16, 6))

plt.subplot(1, 3, 1)
plt.hist(small_df["score"], bins=21, color="blue", alpha=0.7, range=(0.1, 1))
plt.title("Distribution of Small Object Confidence Scores")
plt.xlabel("Confidence Score")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
plt.hist(medium_df["score"], bins=21, color="orange", alpha=0.7, range=(0.1, 1))
plt.title("Distribution of Medium Object Confidence Scores")
plt.xlabel("Confidence Score")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
plt.hist(large_df["score"], bins=21, color="green", alpha=0.7, range=(0.1, 1))
plt.title("Distribution of Large Object Confidence Scores")
plt.xlabel("Confidence Score")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

hist_small, bins_small = np.histogram(small_df["score"], bins=21)

print(
    "Global Minimum Confidence Score for Small Objects:",
    float(bins_small[np.argmin(hist_small)]),
)

hist_medium, bins_medium = np.histogram(medium_df["score"], bins=21)

print(
    "Global Minimum Confidence Score for Medium Objects:",
    float(bins_medium[np.argmin(hist_medium)]),
)

hist_large, bins_large = np.histogram(large_df["score"], bins=21)

print(
    "Global Minimum Confidence Score for Large Objects:",
    float(bins_large[np.argmin(hist_large)]),
)

In [ ]:
def get_bowl_minimum(scores: pl.Series, bins: int = 21, range: tuple = (0, 1)) -> float:
    hist, bin_edges = np.histogram(scores, bins=bins, range=range)

    left_max_index = np.argmax(hist[: len(hist) // 2])
    right_max_index = np.argmax(hist[len(hist) // 2 :]) + len(hist) // 2

    local_min_index = np.argmin(hist[left_max_index:right_max_index]) + left_max_index
    return float(bin_edges[local_min_index])


print(
    "Bowl Minimum Confidence Score for Small Objects:",
    get_bowl_minimum(small_df["score"]),
)
print(
    "Bowl Minimum Confidence Score for Medium Objects:",
    get_bowl_minimum(medium_df["score"]),
)
print(
    "Bowl Minimum Confidence Score for Large Objects:",
    get_bowl_minimum(large_df["score"]),
)